# Assignment 2 research runner

This notebook is a thin launcher for the repository scripts. It does not duplicate model or training logic. A free CPU runtime is sufficient through the Phase 2 handoff verification in the package-and-model-checks cell. The later model-fit and inference pilots require a GPU; availability is not guaranteed. The final application remains local and does not depend on this notebook.

Before running: use `scripts/package_colab_source.py` locally, then copy `artifacts/colab_source.zip` and its `.sha256` sidecar into `MyDrive/GenAI_Assignment_2_source/`. The notebook will not request credentials or print secret values.

In [ ]:
import hashlib
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/GenAI_Assignment_2_source')
ARCHIVE = DRIVE_DIR / 'colab_source.zip'
SIDECAR = DRIVE_DIR / 'colab_source.zip.sha256'
if not ARCHIVE.is_file() or not SIDECAR.is_file():
    raise FileNotFoundError(
        f'Copy colab_source.zip and colab_source.zip.sha256 into {DRIVE_DIR} first.'
    )
expected_hash = SIDECAR.read_text(encoding='utf-8').split()[0]
actual_hash = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
if actual_hash != expected_hash:
    raise ValueError(f'Source archive checksum mismatch: {actual_hash}')
PROJECT_DIR = DRIVE_DIR / f'source_{actual_hash[:12]}'
with zipfile.ZipFile(ARCHIVE) as archive:
    root = PROJECT_DIR.resolve()
    for member in archive.infolist():
        target = (PROJECT_DIR / member.filename).resolve()
        if root not in target.parents and target != root:
            raise ValueError(f'Unsafe source archive path: {member.filename}')
    archive.extractall(PROJECT_DIR)
PROJECT_DIR = PROJECT_DIR / 'Gen_AI_Assignment_2'
if not (PROJECT_DIR / 'scripts' / 'doctor.py').is_file():
    raise FileNotFoundError(f'Project scripts not found under {PROJECT_DIR}')
ACTIVE_DIR = Path(f'/content/Gen_AI_Assignment_2_{actual_hash[:12]}')
ACTIVE_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
COLAB_REPORT = Path('results/environments') / f'colab_{RUN_ID}.json'
print('Verified source:', ARCHIVE)
print('Active runtime workspace:', ACTIVE_DIR)

In [ ]:
import os
import shutil
import subprocess
import sys

os.environ['HF_HOME'] = '/content/.cache/huggingface'
os.environ['HF_HUB_CACHE'] = '/content/.cache/huggingface/hub'
os.environ['PROJECT_RUNTIME_PROFILE'] = 'colab'
shutil.copytree(PROJECT_DIR, ACTIVE_DIR, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('.venv', '.cache', '*.parquet', '*.jsonl', '*.safetensors'))
sys.path.insert(0, str(ACTIVE_DIR))
subprocess.run(
    [sys.executable, 'scripts/doctor.py', '--smoke-test', '--output', str(COLAB_REPORT)],
    cwd=ACTIVE_DIR,
    check=True,
)
with open(ACTIVE_DIR / COLAB_REPORT, encoding='utf-8') as report_file:
    environment_report = report_file.read()
print(environment_report)

## Package and model checks

Colab's existing Torch build is retained. The next cell installs only the pinned shared packages. Restart the runtime if Colab reports that a package update requires it, then rerun the environment doctor.

In [ ]:
import json
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ACTIVE_DIR / 'requirements-colab.txt'), '-r', str(ACTIVE_DIR / 'requirements-dev.txt')], check=True)
subprocess.run([sys.executable, 'scripts/doctor.py', '--smoke-test', '--output', str(COLAB_REPORT)],
               cwd=ACTIVE_DIR, check=True)
pointer = {
    'local_application_environment': 'not copied into Colab workspace',
    'primary_research_environment': str(COLAB_REPORT),
    'status': 'Colab environment inspected; see the linked report for actual allocation.',
}
(ACTIVE_DIR / 'results/environment.json').write_text(
    json.dumps(pointer, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)
subprocess.run([sys.executable, 'scripts/check_tokenizers.py', '--output', 'results/tokenizer_compatibility_colab.json'],
               cwd=ACTIVE_DIR, check=True)
subprocess.run([sys.executable, 'scripts/verify_colab_handoff.py'], cwd=ACTIVE_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ACTIVE_DIR, check=True)

## Fit and inference pilots

The pilot uses one synthetic criterion and no patient dataset rows. Do not start a full training run until model fit, tokenizer compatibility, and the checkpoint restore rehearsal are accepted. Copy each verified output bundle to Drive and download it to the laptop before accepting a long job.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/smoke_models.py', '--output', 'results/feasibility_colab.json'],
               cwd=ACTIVE_DIR, check=True)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/fit_pilot.py', '--steps', '10',
                '--gradient-accumulation', '8', '--max-seq-len', '1024',
                '--output', 'results/fit_pilot_colab.json'],
               cwd=ACTIVE_DIR, check=True)

In [ ]:
from datetime import datetime, timezone
import json
import shutil
from src.clinical_match.checkpointing import (
    create_verified_bundle,
    restore_verified_bundle,
    verify_bundle,
)

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
snapshot_source = ACTIVE_DIR / 'results'
snapshot_destination = PROJECT_DIR / 'verified_bundles'
bundle = create_verified_bundle(
    snapshot_source,
    snapshot_destination,
    f'phase01-{stamp}',
    {'runtime': 'colab', 'purpose': 'phase-0-1-pilot-results'},
)
print('Verified persistent bundle:', bundle)
restored = restore_verified_bundle(bundle, ACTIVE_DIR / 'verified_restore')
if verify_bundle(restored)['files'] != verify_bundle(bundle)['files']:
    raise ValueError('Restored bundle inventory does not match the Drive copy.')
print('Verified local restore from Drive bundle:', restored)
print('Download this bundle to the local D: project before ending the runtime.')

## Phase 4: LoRA training and adapter validation

Run the real-model overfit smoke first, then the interrupted/resumed rehearsal. Start the full run only after both pass. The trainer checks the canonical manifest and data hashes, scores validation with the frozen prompt/parser, and publishes verified bundles to Drive when `--drive-dir` is set. Never evaluate the sealed test split here.


In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, 'scripts/train.py', '--overfit-smoke',
    '--overfit-examples', '12', '--max-steps', '40',
    '--run-name', f'phase04_overfit_{RUN_ID}',
], cwd=ACTIVE_DIR, check=True)

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, 'scripts/rehearse_resume.py',
    '--max-steps', '4', '--interrupt-at', '2',
], cwd=ACTIVE_DIR, check=True)

## Full Phase 4 training

3 epochs, LoRA r=8, AdamW 2e-4, FP16, accumulation 8. Checkpoints are verified locally and copied to `MyDrive/GenAI_Assignment_2_checkpoints/<run>/checkpoints/`. Keep the browser tab alive; if Colab disconnects, restart from the newest resume bundle with `--resume`.


In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, 'scripts/train.py',
    '--run-name', f'phase04_lora_{RUN_ID}',
    '--drive-dir', '/content/drive/MyDrive/GenAI_Assignment_2_checkpoints',
], cwd=ACTIVE_DIR, check=True)

## Verify and download the Phase 4 export

This re-verifies the export bundle, restores it to a fresh directory, then writes a zip plus SHA-256 sidecar to Drive. Download that zip locally into `D:\\Gen_AI_Assignment_2\\artifacts\\phase04\\` and verify it there before claiming Phase 4 complete.


In [ ]:
import hashlib, json, shutil
from datetime import datetime, timezone
from src.clinical_match.checkpointing import restore_verified_bundle, verify_bundle

run_name = f'phase04_lora_{RUN_ID}'
export_root = ACTIVE_DIR / 'checkpoints' / run_name / 'exports'
latest = json.loads((export_root / 'latest.json').read_text(encoding='utf-8'))
bundle = export_root / latest['bundle_name']
verify_bundle(bundle)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
restored = restore_verified_bundle(bundle, ACTIVE_DIR / 'verified_restore' / f'phase04_{stamp}')
if verify_bundle(restored)['files'] != verify_bundle(bundle)['files']:
    raise ValueError('Restored Phase 4 bundle inventory differs from the source bundle')
destination = DRIVE_DIR / f'phase04-{RUN_ID}.zip'
archive = shutil.make_archive(str(destination.with_suffix('')), 'zip', root_dir=bundle)
digest = hashlib.sha256(Path(archive).read_bytes()).hexdigest()
(DRIVE_DIR / f'phase04-{RUN_ID}.zip.sha256').write_text(f'{digest}  {Path(archive).name}\n', encoding='utf-8')
print('Verified bundle:', bundle)
print('Download archive:', archive)
print('SHA-256:', digest)
print('Download this zip to D:\\Gen_AI_Assignment_2\\artifacts\\phase04\\ before ending the runtime.')